In [20]:
!pip install langgraph langchain-ollama!pip install -q langgraph langchain langchain-community langchain-ollama duckduckgo-search python-dotenv

ERROR: Invalid requirement: 'langchain-ollama!pip': Expected semicolon (after name with no version specifier) or end
    langchain-ollama!pip
                    ^


In [21]:
from typing import TypedDict

from langgraph.graph import StateGraph, END

from langchain_ollama import ChatOllama

from langchain_community.tools import DuckDuckGoSearchResults

import sqlite3

In [22]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.3
)

In [23]:
response = llm.invoke("Hello")
print(response.content)

How can I assist you today?


In [24]:
conn = sqlite3.connect("chatbot.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS knowledge(
question TEXT,
answer TEXT
)
""")

conn.commit()

print("Database Created Successfully")

Database Created Successfully


In [25]:
cursor.execute("""
INSERT INTO knowledge VALUES(
'What is AI?',
'Artificial Intelligence is the simulation of human intelligence by machines.'
)
""")

cursor.execute("""
INSERT INTO knowledge VALUES(
'Who developed Python?',
'Python was developed by Guido van Rossum.'
)
""")

conn.commit()

print("Sample Data Inserted")

Sample Data Inserted


In [26]:
class AgentState(TypedDict):
    question: str
    answer: str
    found: bool

In [27]:
def database_query(state):

    question = state["question"]

    cursor.execute(
        "SELECT answer FROM knowledge WHERE question=?",
        (question,)
    )

    result = cursor.fetchone()

    if result:

        return {
            "answer": result[0],
            "found": True
        }

    return {
        "answer": "",
        "found": False
    }

In [28]:
search = DuckDuckGoSearchResults()

In [29]:
def web_search(state):

    question = state["question"]

    result = search.invoke(question)

    return {
        "answer": result,
        "found": True
    }

In [30]:
def response_generator(state):

    prompt = f"""
Question:
{state['question']}

Information:
{state['answer']}

Generate a helpful answer.
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [31]:
def router(state):

    if state["found"]:

        return "response"

    return "search"

In [32]:
graph = StateGraph(AgentState)

graph.add_node("database", database_query)

graph.add_node("search", web_search)

graph.add_node("response", response_generator)

graph.set_entry_point("database")

graph.add_conditional_edges(
    "database",
    router,
    {
        "response": "response",
        "search": "search"
    }
)

graph.add_edge("search", "response")

graph.add_edge("response", END)

app = graph.compile()

print("Graph Created Successfully")

Graph Created Successfully


In [33]:
question = input("Ask your question: ")

result = app.invoke({
    "question": question,
    "answer": "",
    "found": False
})

print("\nAnswer:\n")
print(result["answer"])


Answer:

Hi there! It sounds like you're looking for some fun and creative ways to express "hi" to someone. I've got you covered!

If you want to add a cute touch to your conversations, you can use popular animated GIFs from Tenor or other websites. There are many adorable options available, such as the cartoon bunny saying "hi" on a table.

Alternatively, you can send a thoughtful ecard or card with a message like "Hi, How Ya Doing?" to brighten someone's day.

If you're looking for something more musical, there's even a fun song called "Hi, Hello..." that's perfect for kids and language learners alike!

Lastly, if you want to add some visual humor to your messages, you can check out GIFER's collection of funny "hi" GIFs. They regularly update with new animations about all sorts of things.

Which one of these options sounds like just what you need to say "hi" in a fun and creative way?
